In [ ]:
# Customer Churn Prediction System - Part 2: Feature Engine...
import pandas as pd
import numpy as np
import os
import sys

sys.path.append(os.path.abspath('..'))

In [ ]:
# 1. Load Processed Clean Data
from src.data_preprocessing import ChurnDataPreprocessor

cleaned_path = "../data/processed/cleaned_churn.csv"
preprocessor = ChurnDataPreprocessor(
    raw_data_path="../data/raw/Telco_customer_churn.csv",
    processed_data_path=cleaned_path
)
df = preprocessor.run_pipeline()
print(f"Loaded clean data shape: {df.shape}")

In [ ]:
# 2. Feature Creation
def bin_tenure(months):
    if months <= 12:
        return 'New Customer'
    elif months <= 36:
        return 'Medium Customer'
    else:
        return 'Loyal Customer'

df['Tenure Group'] = df['Tenure Months'].apply(bin_tenure)
print(df['Tenure Group'].value_counts())

In [ ]:
# 2. Feature Creation - Part 2
df['Average Monthly Spend'] = np.where(
    df['Tenure Months'] > 0,
    df['Total Charges'] / df['Tenure Months'],
    df['Monthly Charges']
)
df[['Tenure Months', 'Monthly Charges', 'Total Charges', 'Average Monthly Spend']].head(10)

In [ ]:
# 2. Feature Creation - Part 3
service_cols = ['Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies']
df['Customer Service Count'] = df[service_cols].apply(
    lambda row: sum(1 for val in row if str(val).strip().lower() == 'yes'), axis=1
)
print(df['Customer Service Count'].value_counts())

In [ ]:
# Let's see if the new features are predictive of Churn.
print("Churn rate by Tenure Group:")
print(df.groupby('Tenure Group')['Churn Value'].mean())

print("\nAverage Customer Service Count for Churn vs Non-Churn:")
print(df.groupby('Churn Value')['Customer Service Count'].mean())

In [ ]:
# 3. Scale and Encode using the pipeline script
from src.feature_engineering import ChurnFeatureEngineer

X = df.drop(columns=['Churn Value'])
y = df['Churn Value']

fe = ChurnFeatureEngineer(encoder_path="../models/encoder.joblib", scaler_path="../models/scaler.joblib")

X_engineered = fe.create_features(X)
X_transformed = fe.fit_transform(X_engineered)
fe.save_assets()

print(f"Transformed data shape: {X_transformed.shape}")
print("Sample engineered columns:")
print(X_transformed.columns[:10])

In [ ]:
# 3. Scale and Encode using the pipeline script - Part 2
X_transformed.describe().round(3)